# PySpark DataFrame Data Engineering Project — Energy Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `meters.csv`, `energy_usage.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("meters.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("energy_usage.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select meter_id and city from the first DataFrame.

In [ ]:
df1.select('meter_id','city').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where bill_amount is greater than its average.

In [ ]:
avg_value = df2.select(avg('bill_amount').alias('avg')).first()['avg']
df2.filter(col('bill_amount') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on bill_amount using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('bill_amount') >= avg('bill_amount').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in city and connection_type with 'Unknown'.

In [ ]:
df1.fillna({'city': 'Unknown'}).show(5)
df2.fillna({'connection_type': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using meter_id and meter_id.

In [ ]:
joined = df1.join(df2, df1['meter_id'] == df2['meter_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['meter_id'] == df2['meter_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average bill_amount by connection_type.

In [ ]:
df2.groupBy('connection_type').agg(count('*').alias('records'), avg('bill_amount').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total bill_amount by connection_type.

In [ ]:
df2.groupBy('connection_type').agg(sum('bill_amount').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum bill_amount in each connection_type.

In [ ]:
df2.groupBy('connection_type').agg(max('bill_amount').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum bill_amount in each connection_type.

In [ ]:
df2.groupBy('connection_type').agg(min('bill_amount').alias('min_value')).show()

## Q14. Having Condition

Show connection_type groups whose total bill_amount is greater than the overall average bill_amount.

In [ ]:
overall = df2.select(avg('bill_amount').alias('v')).first()['v']
df2.groupBy('connection_type').agg(sum('bill_amount').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by bill_amount.

In [ ]:
df2.orderBy(desc('bill_amount')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by bill_amount.

In [ ]:
df2.orderBy(asc('bill_amount')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by bill_amount within each connection_type.

In [ ]:
w = Window.partitionBy('connection_type').orderBy(desc('bill_amount'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of bill_amount within each connection_type.

In [ ]:
w = Window.partitionBy('connection_type').orderBy('reading_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('bill_amount').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's bill_amount with the previous record in its connection_type.

In [ ]:
w = Window.partitionBy('connection_type').orderBy('reading_date')
df2.withColumn('previous_value', lag('bill_amount').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each connection_type's contribution to total bill_amount.

In [ ]:
tot = df2.select(sum('bill_amount').alias('t')).first()['t']
df2.groupBy('connection_type').agg(sum('bill_amount').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from reading_date.

In [ ]:
df2.withColumn('year', year('reading_date')).withColumn('month', month('reading_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average bill_amount by month.

In [ ]:
df2.withColumn('month', date_format('reading_date','yyyy-MM')).groupBy('month').agg(sum('bill_amount').alias('total'), avg('bill_amount').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with connection_type as rows and year of reading_date as columns.

In [ ]:
df2.withColumn('year', year('reading_date')).groupBy('connection_type').pivot('year').agg(sum('bill_amount')).show()

## Q24. Conditional KPI

Count records where bill_amount is above the category average.

In [ ]:
w=Window.partitionBy('connection_type')
df2.withColumn('cat_avg',avg('bill_amount').over(w)).filter(col('bill_amount')>col('cat_avg')).groupBy('connection_type').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total bill_amount by city.

In [ ]:
df1.join(df2, df1['meter_id']==df2['meter_id']).groupBy(df1['city']).agg(sum(df2['bill_amount']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank city by total bill_amount.

In [ ]:
r=df1.join(df2,df1['meter_id']==df2['meter_id']).groupBy(df1['city']).agg(sum(df2['bill_amount']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for bill_amount.

In [ ]:
df2.withColumn('segment',when(col('bill_amount')<1000,'Low').when(col('bill_amount')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative bill_amount values.

In [ ]:
df2.filter(col('bill_amount')<0).count()

## Q29. Final Business Report

Create a final report by connection_type with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('connection_type').agg(count('*').alias('records'),sum('bill_amount').alias('total'),avg('bill_amount').alias('average'),min('bill_amount').alias('minimum'),max('bill_amount').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project